# CSPF stage R: flip rate on the reliably labelled round-3 pairs
Run all (Runtime > Run all). CPU is enough (about 10-15 minutes). Needs `ALL_MODELS_combined_FIXED.csv` and `Round3_for_join_WITH_TEXT.csv` in the paper2 folder of your Drive. Send back `CSPF_stageR_bundle.zip`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
LABELS = f"{BASE}/Round3_for_join_WITH_TEXT.csv"      # file 'Round3_for_join_WITH_TEXT.csv' from the assistant; put it in BASE
OUT = f"{BASE}/results_FIXED/reviewer_R"; B_REP = 1000
import os, warnings, zipfile, glob, html, re; warnings.filterwarnings("ignore"); os.makedirs(OUT, exist_ok=True)
import numpy as np, pandas as pd
def norm(t): return re.sub(r"\s+", " ", html.unescape(str(t))).strip().lower()
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist()
MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
meta = ["source_dataset","review_idx","perturbed_review","perturbation_family","severity"]
use = meta + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
parts, Fl = [], []
for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
    Fl.append(np.stack([(ch[f"{m}_orig_label"].values != ch[f"{m}_pert_label"].values) for m in MODELS], 1).astype(np.int8)); parts.append(ch[meta])
df = pd.concat(parts, ignore_index=True); F = np.concatenate(Fl); del parts, Fl
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str); df["nt"] = df.perturbed_review.map(norm); df["pair_id"] = np.arange(len(df))
print(len(df), "pairs;", df.key.nunique(), "reviews;", len(MODELS), "models")

In [ ]:
# ── R1: flip rate by the majority label of the three round-3 annotators (kappa 0.72), matched to the prediction table ──
L = pd.read_csv(LABELS, encoding="utf-8-sig"); L["source_dataset"] = L.source_dataset.astype(str); L["nt"] = L.perturbed_review.map(norm); L["review_idx"] = L.review_idx.astype(df.review_idx.dtype)
lk = df[["source_dataset","review_idx","perturbation_family","nt","key","pair_id"]].drop_duplicates(["source_dataset","review_idx","perturbation_family","nt"])
j = L.merge(lk, on=["source_dataset","review_idx","perturbation_family","nt"], how="inner"); print(len(j), "of", len(L), "round-3 pairs matched to the prediction table")
assert len(j) >= 0.8*len(L), "too few matches: check LABELS file and FIXED_CSV"
Fi = F[j.pair_id.values]; mm = j.meaning_maj.values; PROP = {"P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"}
isprop = j.perturbation_family.isin(PROP).values
groups = {"all matched": np.ones(len(j), bool), "meaning yes (majority)": mm=="yes", "meaning partial (majority)": mm=="partial", "meaning no (majority)": mm=="no",
          "yes + sentiment yes": (mm=="yes")&(j.sentiment_maj.values=="yes"), "yes, propositional families": (mm=="yes")&isprop, "partial/no, propositional families": (mm!="yes")&isprop, "partial/no, modulating families": (mm!="yes")&~isprop, "yes, modulating families": (mm=="yes")&~isprop}
rng = np.random.default_rng(0); uk, ki = np.unique(j.key.values, return_inverse=True); rows = []
for nm, m in groups.items():
    if m.sum() == 0: continue
    v = []
    for _ in range(B_REP):
        c = np.bincount(rng.integers(0, len(uk), len(uk)), minlength=len(uk))[ki][m]
        if c.sum(): v.append((c*Fi[m].mean(1)).sum()/c.sum())
    rows.append(dict(group=nm, pairs=int(m.sum()), FR=round(Fi[m].mean(),4), ci_lo=round(np.percentile(v,2.5),4), ci_hi=round(np.percentile(v,97.5),4)))
R1 = pd.DataFrame(rows); print(R1.to_string(index=False)); R1.to_csv(f"{OUT}/R1_FR_by_round3_label.csv", index=False)

In [ ]:
# ── R2: per-model flip rate on the reliably verified pairs, ranking against the full corpus, Cochran's Q ──
from scipy.stats import spearmanr
import subprocess, sys; subprocess.run([sys.executable,"-m","pip","-q","install","statsmodels"],check=False)
from statsmodels.stats.contingency_tables import cochrans_q
full = F.mean(0); mv = groups["meaning yes (majority)"]; ma = groups["all matched"]
pm = pd.DataFrame({"model": MODELS, "FR_full_corpus": full.round(4), "FR_verified_yes": Fi[mv].mean(0).round(4), "FR_matched_all": Fi[ma].mean(0).round(4), "FR_partial": Fi[groups["meaning partial (majority)"]].mean(0).round(4)})
print(pm.to_string(index=False)); pm.to_csv(f"{OUT}/R2_per_model.csv", index=False)
res = dict(n_verified=int(mv.sum()), spearman_verified_vs_full=spearmanr(Fi[mv].mean(0), full)[0], spearman_matched_vs_full=spearmanr(Fi[ma].mean(0), full)[0])
if mv.sum() >= 30:
    q = cochrans_q(Fi[mv]); res.update(Q=q.statistic, df=q.df, p=q.pvalue)
print(res); pd.Series(res).to_csv(f"{OUT}/R2_tests.csv")
with zipfile.ZipFile(f"{OUT}/CSPF_stageR_bundle.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.csv"): z.write(f, os.path.basename(f))
print("send me:", f"{OUT}/CSPF_stageR_bundle.zip")